# Fix Missing Tables — Targeted Reload

Reloads ONLY the 3 tables that are empty/incomplete:
1. **billing.usage** — currently 4,000 rows (only to May 2023). Needs full 3 years.
2. **compute.clusters** — currently 0 rows. Needs 200.
3. **lakeflow.jobs** — currently 0 rows. Needs 500.

**Does NOT touch** the other 30+ tables that are already loaded.

In [ ]:
import os, time, random
from datetime import datetime, timedelta
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

# Config
try:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    HOST = ctx.apiUrl().get()
    TOKEN = ctx.apiToken().get()
except:
    HOST = os.environ.get('DATABRICKS_HOST', '')
    TOKEN = os.environ.get('DATABRICKS_TOKEN', '')

WAREHOUSE_ID = os.environ.get('DATABRICKS_WAREHOUSE_ID', '21f5bd20b7f44a51')
client = WorkspaceClient(host=HOST, token=TOKEN)

def run_sql(label, sql, timeout_sec=300):
    resp = client.statement_execution.execute_statement(
        warehouse_id=WAREHOUSE_ID, statement=sql, wait_timeout='50s')
    deadline = time.time() + timeout_sec
    while resp.status.state in (StatementState.PENDING, StatementState.RUNNING):
        if time.time() > deadline:
            print(f'  ⏱ TIMEOUT: {label}'); return False
        time.sleep(3)
        resp = client.statement_execution.get_statement(resp.statement_id)
    if resp.status.state == StatementState.SUCCEEDED:
        print(f'  ✅ {label}'); return True
    err = getattr(resp.status.error, 'message', str(resp.status.state))
    print(f'  ⚠️ SKIP ({err[:100]}): {label}'); return False

def sql_str(s): return s.replace("'", "''")
def chunk_list(lst, size):
    for i in range(0, len(lst), size): yield lst[i:i+size]

print(f'Connected to {HOST}, warehouse {WAREHOUSE_ID}')

In [ ]:
# ── Constants (same seeds as enterprise script for consistency) ─────────
ACCOUNT_ID = 'acc-chase-001'
WORKSPACES = [
    ('ws-100001','prod-consumer-banking','us-east-1'),
    ('ws-100002','prod-investment-banking','us-east-1'),
    ('ws-100003','prod-risk-analytics','us-west-2'),
    ('ws-100004','prod-data-platform','us-east-1'),
    ('ws-100005','prod-fraud-detection','us-west-2'),
]
DEPARTMENTS = ['Consumer Banking','Investment Banking','Risk Analytics','Fraud Detection',
    'Data Engineering','Data Science','Compliance','Treasury','Marketing Analytics',
    'Credit Risk','Market Risk','Operations','Wealth Management','Card Services','Mortgage']
TEAMS = ['platform-core','etl-pipeline','ml-ops','analytics-eng','feature-store','data-quality',
    'streaming-ingest','reporting','bi-team','quant-research','fraud-ml','aml-detection',
    'credit-scoring','nrt-analytics','data-governance','lakehouse-admin','cost-optimization']
SKU_MAP = {
    'STANDARD_ALL_PURPOSE_COMPUTE':0.55,'PREMIUM_ALL_PURPOSE_COMPUTE':0.70,
    'JOBS_COMPUTE':0.15,'JOBS_LIGHT_COMPUTE':0.10,'SERVERLESS_SQL':0.70,'PRO_SQL':0.55,
    'SERVERLESS_REAL_TIME_INFERENCE':0.07,'GPU_ALL_PURPOSE_COMPUTE':1.50,
    'DLT_CORE_COMPUTE':0.20,'DLT_PRO_COMPUTE':0.25,'DLT_ADVANCED_COMPUTE':0.36,
}
SKU_WEIGHTS = {
    'JOBS_COMPUTE':0.35,'SERVERLESS_SQL':0.20,'PRO_SQL':0.10,
    'STANDARD_ALL_PURPOSE_COMPUTE':0.08,'PREMIUM_ALL_PURPOSE_COMPUTE':0.05,
    'DLT_PRO_COMPUTE':0.07,'DLT_ADVANCED_COMPUTE':0.04,'DLT_CORE_COMPUTE':0.03,
    'GPU_ALL_PURPOSE_COMPUTE':0.04,'SERVERLESS_REAL_TIME_INFERENCE':0.02,'JOBS_LIGHT_COMPUTE':0.02,
}
NODE_TYPES = ['i3.xlarge','i3.2xlarge','i3.4xlarge','i3.8xlarge','r5.xlarge','r5.2xlarge',
    'r5.4xlarge','r5.8xlarge','m5.xlarge','m5.2xlarge','m5.4xlarge',
    'p3.2xlarge','p3.8xlarge','g4dn.xlarge','g4dn.4xlarge']
DBR_VERSIONS = ['12.2.x-scala2.12','13.0.x-scala2.12','13.3.x-scala2.12','14.0.x-scala2.12',
    '14.3.x-scala2.12','15.0.x-scala2.12','15.2.x-scala2.12','15.4.x-scala2.12']
FIRST_NAMES = ['James','Mary','Robert','Patricia','John','Jennifer','Michael','Linda','David',
    'Elizabeth','William','Barbara','Richard','Susan','Joseph','Jessica','Thomas','Sarah',
    'Christopher','Karen','Charles','Lisa','Daniel','Nancy','Matthew','Betty','Anthony',
    'Margaret','Mark','Sandra','Donald','Ashley','Steven','Dorothy','Paul','Kimberly',
    'Andrew','Emily','Joshua','Donna','Kenneth','Michelle','Kevin','Carol','Brian','Amanda',
    'George','Melissa','Timothy','Deborah','Ronald','Stephanie','Edward','Rebecca','Jason',
    'Sharon','Jeffrey','Laura','Ryan','Cynthia','Jacob','Kathleen','Gary','Amy','Nicholas',
    'Angela','Eric','Shirley','Jonathan','Anna','Stephen','Brenda','Larry','Pamela','Justin',
    'Emma','Scott','Nicole','Brandon','Helen','Benjamin','Samantha','Samuel','Katherine',
    'Raymond','Christine','Gregory','Debra','Frank','Rachel','Alexander','Carolyn','Patrick',
    'Janet','Jack','Catherine','Dennis','Maria','Jerry','Heather','Tyler','Diane','Aaron',
    'Ruth','Jose','Julie','Nathan','Olivia','Henry','Joyce','Peter','Virginia','Douglas',
    'Victoria','Zachary','Kelly','Kyle','Lauren','Noah','Christina','Ethan','Joan','Adrian',
    'Evelyn','Aiden','Judith','Dylan','Megan','Priya','Raj','Anita','Vikram','Deepa',
    'Sanjay','Neha','Amit','Wei','Ming','Yuki','Hiro','Jin','Soo','Chen','Li']
LAST_NAMES = ['Smith','Johnson','Williams','Brown','Jones','Garcia','Miller','Davis','Rodriguez',
    'Martinez','Hernandez','Lopez','Gonzalez','Wilson','Anderson','Thomas','Taylor','Moore',
    'Jackson','Martin','Lee','Perez','Thompson','White','Harris','Sanchez','Clark','Ramirez',
    'Lewis','Robinson','Walker','Young','Allen','King','Wright','Scott','Torres','Nguyen',
    'Hill','Flores','Green','Adams','Nelson','Baker','Hall','Rivera','Campbell','Mitchell',
    'Carter','Roberts','Patel','Shah','Kumar','Singh','Gupta','Sharma','Chen','Wang','Li',
    'Zhang','Liu','Yang','Wu','Kim','Park','Cho','Jung','Tanaka','Suzuki','Watanabe',
    "O'Brien",'Murphy','Sullivan','Cohen','Goldberg','Katz']

# Generate users (same seed=42 as enterprise script)
def gen_users(count=1100):
    users, used = [], set()
    random.seed(42)
    for _ in range(count * 2):
        if len(users) >= count: break
        email = f"{random.choice(FIRST_NAMES).lower()}.{random.choice(LAST_NAMES).lower()}@chase.com"
        if email not in used: used.add(email); users.append(email)
    return users

USERS = gen_users(1100)
NOW = datetime(2026, 4, 26)
THREE_YEARS_AGO = NOW - timedelta(days=1095)

random.seed(42)
USER_DEPT = {u: random.choice(DEPARTMENTS) for u in USERS}
USER_TEAM = {u: random.choice(TEAMS) for u in USERS}

print(f'{len(USERS)} users, date range {THREE_YEARS_AGO.date()} → {NOW.date()}')

In [ ]:
# ── Generate cluster + job definitions (needed for billing references) ──
def gen_clusters(count=200):
    random.seed(100)
    clusters = []
    for i in range(count):
        ws = random.choice(WORKSPACES); owner = random.choice(USERS)
        cname = '-'.join([random.choice(['etl','analytics','ml','streaming','adhoc','prod','staging','dev']),
                          random.choice(['pipeline','cluster','compute','workload','processing']),
                          str(random.randint(1,50))])
        clusters.append(dict(ws=ws, id=f'cls-{i+1:04d}', name=cname, owner=owner,
            driver=random.choice(NODE_TYPES[:11]), worker=random.choice(NODE_TYPES[:11]),
            workers=random.choice([2,4,8,16,32]),
            min_w=max(1,random.choice([2,4,8,16,32])//4),
            max_w=random.choice([2,4,8,16,32])*2,
            auto_term=random.choice([60,120,240,0]),
            dbr=random.choice(DBR_VERSIONS), team=USER_TEAM[owner], dept=USER_DEPT[owner],
            days_ago=random.randint(30,1000), deleted=random.random()<0.15,
            security=random.choice(['SINGLE_USER','USER_ISOLATION','NO_ISOLATION'])))
    return clusters

def gen_warehouses(count=30):
    random.seed(200)
    whs = []
    for i in range(count):
        ws = random.choice(WORKSPACES)
        whs.append(dict(ws=ws, id=f'wh-{i+1:04d}',
            name=f"{random.choice(['reporting','bi','adhoc','etl','prod','staging'])}-warehouse-{i+1}",
            type=random.choice(['PRO','CLASSIC','SERVERLESS']),
            size=random.choice(['2X-Small','X-Small','Small','Medium','Large','X-Large','2X-Large']),
            min_c=random.choice([1,1,1,2]), max_c=random.choice([1,2,4,8,16]),
            auto_stop=random.choice([5,10,15,30]), days_ago=random.randint(30,900)))
    return whs

def gen_jobs(count=500):
    random.seed(300)
    jobs = []
    job_types = ['ETL-Daily','ETL-Hourly','ML-Training','ML-Scoring','Report-Generation',
        'Data-Quality','Feature-Engineering','Streaming-Ingest','CDC-Pipeline','Archive-Job',
        'Compliance-Check','AML-Scan','Fraud-Score','Risk-Calc','PnL-Report',
        'Regulatory-Filing','Customer-360','Segmentation','Campaign-Analytics','Real-Time-Alerts']
    schedules = ['0 0 * * * ?','0 0 8 * * ?','0 0 6 * * ?','0 30 7 * * ?',
                 '0 0 0 * * ?','0 0 */4 * * ?','0 0 9 ? * MON']
    for i in range(count):
        ws = random.choice(WORKSPACES); creator = random.choice(USERS)
        jtype = random.choice(job_types)
        jobs.append(dict(ws=ws, id=f'job-{i+1:05d}',
            name=f"{jtype}-{USER_DEPT[creator].lower().replace(' ','-')}-{random.randint(1,99)}",
            creator=creator, schedule=random.choice(schedules),
            days_ago=random.randint(10,1000), deleted=random.random()<0.08,
            team=USER_TEAM[creator], env=random.choice(['prod','staging','dev'])))
    return jobs

CLUSTERS = gen_clusters()
WAREHOUSES = gen_warehouses()
JOBS = gen_jobs()
print(f'Clusters: {len(CLUSTERS)}, Warehouses: {len(WAREHOUSES)}, Jobs: {len(JOBS)}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# TABLE 1: billing.usage — DROP + recreate with full 3 years
# ══════════════════════════════════════════════════════════════════════════
print('🔄 Rebuilding billing.usage (3 years, ~$5M/year)...')

run_sql('drop billing.usage', 'DROP TABLE IF EXISTS workspace.mock_system_billing.usage')
run_sql('create billing.usage', '''
CREATE TABLE workspace.mock_system_billing.usage (
  record_id STRING, account_id STRING, workspace_id STRING, sku_name STRING, cloud STRING,
  usage_start_time TIMESTAMP, usage_end_time TIMESTAMP, usage_date DATE,
  custom_tags MAP<STRING, STRING>, usage_unit STRING, usage_quantity DOUBLE,
  usage_type STRING, billing_origin_product STRING, record_type STRING, ingestion_date DATE,
  identity_metadata STRUCT<run_as: STRING, created_by: STRING>,
  usage_metadata STRUCT<cluster_id: STRING, warehouse_id: STRING, job_id: STRING,
    job_run_id: STRING, dlt_pipeline_id: STRING, notebook_id: STRING,
    endpoint_name: STRING, endpoint_id: STRING, run_id: STRING>
)
''')

random.seed(42)
rows = []
rc = 0
yearly_daily_target = {0: 9600, 1: 12300, 2: 15000}
current = THREE_YEARS_AGO

while current < NOW:
    yi = min(2, (current - THREE_YEARS_AGO).days // 365)
    dt = yearly_daily_target[yi]
    dow = current.weekday()
    if dow >= 5: dt = int(dt * 0.4)
    if current.day >= 28: dt = int(dt * 1.3)
    if current.month in (3,6,9,12) and current.day >= 25: dt = int(dt * 1.5)

    for sku, weight in SKU_WEIGHTS.items():
        sb = int(dt * weight)
        if sb < 1: continue
        price = SKU_MAP[sku]
        total_dbu = sb / price
        nr = random.randint(5, 25)
        for _ in range(nr):
            rc += 1
            user = random.choice(USERS); ws = random.choice(WORKSPACES)
            team = USER_TEAM[user]; dept = USER_DEPT[user]
            dbu = round(total_dbu / nr * random.uniform(0.5, 1.5), 2)
            ho = random.randint(0, 23); dm = random.randint(10, 180)
            us = current.replace(hour=ho, minute=0, second=0)
            ue = us + timedelta(minutes=dm)
            cid=wid=jid=jrid=dltid=nbid=epn=epi=rid=''
            if 'ALL_PURPOSE' in sku or 'GPU' in sku:
                cid = random.choice(CLUSTERS)['id']; nbid = f'nb-{random.randint(1,5000)}'
            elif 'SQL' in sku: wid = random.choice(WAREHOUSES)['id']
            elif 'JOBS' in sku: j=random.choice(JOBS); jid=j['id']; jrid=f'run-{rc}'
            elif 'DLT' in sku: dltid = f'dlt-{random.randint(1,100):03d}'
            elif 'INFERENCE' in sku:
                epn = random.choice(['fraud-scorer','credit-model','recommender','nlp-classifier'])
                epi = f'ep-{random.randint(1,20):03d}'
            bp = 'INTERACTIVE' if 'ALL_PURPOSE' in sku else ('SQL' if 'SQL' in sku else ('JOBS' if 'JOBS' in sku else ('DLT' if 'DLT' in sku else 'SERVING')))
            rows.append(
                f"('r-{rc:08d}','{ACCOUNT_ID}','{ws[0]}','{sku}','AWS',"
                f"'{us.strftime('%Y-%m-%d %H:%M:%S')}','{ue.strftime('%Y-%m-%d %H:%M:%S')}',"
                f"'{current.strftime('%Y-%m-%d')}',map('team','{sql_str(team)}','department','{sql_str(dept)}'),'DBU',{dbu},'{bp}',"
                f"'{bp}','ORIGINAL','{current.strftime('%Y-%m-%d')}',"
                f"named_struct('run_as','{sql_str(user)}','created_by','{sql_str(user)}'),"
                f"named_struct('cluster_id','{cid}','warehouse_id','{wid}','job_id','{jid}','job_run_id','{jrid}',"
                f"'dlt_pipeline_id','{dltid}','notebook_id','{nbid}','endpoint_name','{epn}','endpoint_id','{epi}','run_id','{rid}'))")
    current += timedelta(days=1)

print(f'Generated {len(rows)} billing rows. Inserting in chunks of 500...')
total_chunks = (len(rows) // 500) + 1
for i, chunk in enumerate(chunk_list(rows, 500)):
    vals = ',\n'.join(chunk)
    run_sql(f'billing chunk {i+1}/{total_chunks}',
            f'INSERT INTO workspace.mock_system_billing.usage VALUES\n{vals}')
print(f'\n✅ billing.usage: {len(rows)} rows loaded')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# TABLE 2: compute.clusters — DROP + recreate with 200 clusters
# ══════════════════════════════════════════════════════════════════════════
print('🔄 Rebuilding compute.clusters (200 clusters)...')

run_sql('drop clusters', 'DROP TABLE IF EXISTS workspace.mock_system_compute.clusters')
run_sql('create clusters', '''
CREATE TABLE workspace.mock_system_compute.clusters (
  account_id STRING, workspace_id STRING, cluster_id STRING, cluster_name STRING,
  owned_by STRING, create_time TIMESTAMP, delete_time TIMESTAMP,
  driver_node_type STRING, worker_node_type STRING, worker_count BIGINT,
  min_autoscale_workers BIGINT, max_autoscale_workers BIGINT,
  auto_termination_minutes BIGINT, enable_elastic_disk BOOLEAN,
  tags MAP<STRING, STRING>, cluster_source STRING, dbr_version STRING,
  change_time TIMESTAMP, change_date DATE, data_security_mode STRING
)
''')

cl_rows = []
for c in CLUSTERS:
    ws = c['ws']
    cd = (NOW - timedelta(days=c['days_ago'])).strftime('%Y-%m-%d %H:%M:%S')
    dt_str = f"cast('{(NOW - timedelta(days=random.randint(1, c['days_ago']))).strftime('%Y-%m-%d %H:%M:%S')}' as TIMESTAMP)" if c['deleted'] else 'cast(null as TIMESTAMP)'
    chd = (NOW - timedelta(days=random.randint(1,30))).strftime('%Y-%m-%d')
    cl_rows.append(
        f"('{ACCOUNT_ID}','{ws[0]}','{c['id']}','{c['name']}','{sql_str(c['owner'])}',"
        f"'{cd}',{dt_str},'{c['driver']}','{c['worker']}',{c['workers']},{c['min_w']},{c['max_w']},"
        f"{c['auto_term']},true,map('team','{c['team']}','department','{c['dept']}'),'API','{c['dbr']}',"
        f"'{chd} 10:00:00','{chd}','{c['security']}')")

run_sql('insert clusters', f'INSERT INTO workspace.mock_system_compute.clusters VALUES\n' + ',\n'.join(cl_rows))
print(f'\n✅ compute.clusters: {len(cl_rows)} rows loaded')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# TABLE 3: lakeflow.jobs — DROP + recreate with 500 jobs
# ══════════════════════════════════════════════════════════════════════════
print('🔄 Rebuilding lakeflow.jobs (500 jobs)...')

run_sql('drop jobs', 'DROP TABLE IF EXISTS workspace.mock_system_lakeflow.jobs')
run_sql('create jobs', '''
CREATE TABLE workspace.mock_system_lakeflow.jobs (
  account_id STRING, workspace_id STRING, job_id STRING, name STRING,
  creator_user_name STRING, run_as_user_name STRING, tags MAP<STRING, STRING>,
  schedule STRUCT<quartz_cron_expression: STRING, pause_status: STRING>,
  created_time TIMESTAMP, change_time TIMESTAMP, delete_time TIMESTAMP
)
''')

j_rows = []
for j in JOBS:
    ws = j['ws']
    cr = (NOW - timedelta(days=j['days_ago'])).strftime('%Y-%m-%d %H:%M:%S')
    ch = (NOW - timedelta(days=random.randint(1,30))).strftime('%Y-%m-%d %H:%M:%S')
    dt = f"cast('{(NOW - timedelta(days=random.randint(1,j['days_ago']))).strftime('%Y-%m-%d %H:%M:%S')}' as TIMESTAMP)" if j['deleted'] else 'cast(null as TIMESTAMP)'
    pa = 'UNPAUSED' if random.random()<0.85 else 'PAUSED'
    j_rows.append(
        f"('{ACCOUNT_ID}','{ws[0]}','{j['id']}','{sql_str(j['name'])}',"
        f"'{sql_str(j['creator'])}','{sql_str(j['creator'])}',"
        f"map('Env','{j['env']}','team','{j['team']}'),"
        f"named_struct('quartz_cron_expression','{j['schedule']}','pause_status','{pa}'),"
        f"'{cr}','{ch}',{dt})")

for i, chunk in enumerate(chunk_list(j_rows, 200)):
    run_sql(f'jobs chunk {i+1}', f'INSERT INTO workspace.mock_system_lakeflow.jobs VALUES\n' + ',\n'.join(chunk))
print(f'\n✅ lakeflow.jobs: {len(j_rows)} rows loaded')

In [ ]:
# ── Verification ─────────────────────────────────────────────────────────
print('\n' + '='*60)
print('🎉 TARGETED FIX COMPLETE')
print('='*60)

for tbl, col in [
    ('workspace.mock_system_billing.usage', 'usage_date'),
    ('workspace.mock_system_compute.clusters', None),
    ('workspace.mock_system_lakeflow.jobs', None),
]:
    if col:
        q = f"SELECT COUNT(*) as cnt, MIN({col}) as mn, MAX({col}) as mx FROM {tbl}"
    else:
        q = f"SELECT COUNT(*) as cnt FROM {tbl}"
    resp = client.statement_execution.execute_statement(
        warehouse_id=WAREHOUSE_ID, statement=q, wait_timeout='30s')
    deadline = time.time() + 60
    while resp.status.state in (StatementState.PENDING, StatementState.RUNNING):
        if time.time() > deadline: break
        time.sleep(2)
        resp = client.statement_execution.get_statement(resp.statement_id)
    if resp.status.state == StatementState.SUCCEEDED and resp.result and resp.result.data_array:
        row = resp.result.data_array[0]
        if col:
            print(f'  {tbl}: {row[0]} rows, {row[1]} → {row[2]}')
        else:
            print(f'  {tbl}: {row[0]} rows')
    else:
        print(f'  {tbl}: could not verify')

print('\n✅ All other tables untouched. Refresh the app to see data.')